# FastAPI backend — Smoke notebook

Vérification manuelle du backend `api/`. Regroupe les tests à faire après chaque PR R4 qui touche à l'API REST ou WebSocket.

**Comment utiliser :**

1. Dans un shell séparé : `docker compose -f docker-compose.dev.yml up -d postgres redis` puis `.\scripts\run_api.ps1` (Windows) ou `bash scripts/run_api.sh` (Linux/macOS).
2. Ici : `Kernel > Restart & Run All`.
3. Avant chaque section, lis la cellule markdown **"Ce que tu dois tester"**.

**Prérequis une seule fois :**
- venv avec `fastapi`, `uvicorn`, `httpx` installés
- Postgres + Redis up

## 0. Setup — client httpx partagé

### Ce que tu dois tester
Le client peut joindre uvicorn sur `localhost:8000`.

**Succès attendu** : `root OK : 200` affiché (via /openapi.json). Pas d'exception.

**Alerte si tu vois** : `ConnectionRefused` → uvicorn n'est pas lancé.

In [ ]:
import httpx

BASE = "http://localhost:8000"
c = httpx.Client(base_url=BASE, timeout=5.0)

r = c.get("/openapi.json")
print("root OK :", r.status_code)
print("title   :", r.json()["info"]["title"], r.json()["info"]["version"])

## 1. Scaffold — `/docs`, `/openapi.json`, `/metrics` (R4 PR #1)

### Ce que tu dois tester
Les endpoints generic FastAPI répondent + `/metrics` renvoie du Prometheus text.

**Succès attendu :** `/docs` → 200 (HTML Swagger), `/openapi.json` → 200 (JSON), `/metrics` → 200 contenant `fxvol_http_request_duration_seconds`.

In [ ]:
for path in ("/docs", "/openapi.json", "/metrics"):
    r = c.get(path)
    print(f"{path:<20} → {r.status_code}  ({len(r.content)} bytes)")

assert "fxvol_http_request_duration_seconds" in c.get("/metrics").text
print("OK — TimingMiddleware actif")

## 2. Health endpoints — `/api/v1/health` + `/extended` (R4 PR #2)

### Ce que tu dois tester
Liveness simple + readiness avec check Redis + DB + heartbeats des 3 engines.

**Succès attendu :** `/health` → `{"status": "OK"}` toujours ; `/health/extended` → status `OK` si `app.py` pousse les heartbeats, `DEGRADED` sinon.

**Alerte si tu vois** : `redis: "DOWN"` ou `database: "DOWN"` → Redis/Postgres pas up.

In [ ]:
import json

r = c.get("/api/v1/health")
print("/health :", r.status_code, r.json())

r = c.get("/api/v1/health/extended")
print("\n/health/extended :", r.status_code)
print(json.dumps(r.json(), indent=2))

## 3. Pricing — `/api/v1/{price,greeks,iv}` (R4 PR #3)

### Ce que tu dois tester
Les 3 endpoints BS (Black-Scholes) fonctionnent : prix, greeks bundle, implied vol par inversion.

**Succès attendu :**
- `/price` sur un ATM call EURUSD 30d 7.5% → prix > 0 (typiquement ~0.0091)
- Put-Call parity sur ATM (F=K) → call price == put price (à 1e-9 près, rates=0)
- `/greeks` → payload avec price + delta + gamma + vega + theta, delta ATM ≈ 0.5
- `/iv` round-trip : price → iv → retrouve la vol d'entrée (à 1e-4 près)
- Validation : `volatility=0` ou `option_type="BLABLA"` → 422

**Alerte si tu vois :**
- Prix négatif → bug dans bs_pricer, regarder la trace
- `/iv` renvoie 422 sur le round-trip → le price computé est hors bracket [1e-6, 5.0], probablement un bug de conversion F/K/T

In [ ]:
# ATM EURUSD 30d 7.5% vol — inputs standards du trading desk
atm = {
    "spot": 1.0850, "strike": 1.0850, "maturity_days": 30,
    "volatility": 0.075, "option_type": "CALL",
}

# Price
r = c.post("/api/v1/price", json=atm)
price_call = r.json()["price"]
print(f"/price CALL ATM     → {price_call:.6f}  (status {r.status_code})")

# Put-Call parity
r = c.post("/api/v1/price", json={**atm, "option_type": "PUT"})
price_put = r.json()["price"]
print(f"/price PUT  ATM     → {price_put:.6f}")
print(f"parity diff         → {abs(price_call - price_put):.2e}  (doit etre ~0 pour F=K)")

In [ ]:
# Greeks bundle
r = c.post("/api/v1/greeks", json=atm)
greeks = r.json()
print(f"/greeks ATM         → {r.status_code}")
for k, v in greeks.items():
    print(f"  {k:<8} : {v:.6f}")
print(f"\ndelta ATM ~ 0.5 ?   → {greeks['delta']:.3f}")

In [ ]:
# Round-trip : price avec sigma=0.075 → renvoyer ce price dans /iv → doit retrouver 0.075
iv_req = {
    "spot": atm["spot"], "strike": atm["strike"],
    "maturity_days": atm["maturity_days"],
    "market_price": price_call,
    "option_type": atm["option_type"],
}
r = c.post("/api/v1/iv", json=iv_req)
iv = r.json()["implied_volatility"]
print(f"/iv round-trip      → {iv:.6f}  (input vol etait 0.075000)")
print(f"error               → {abs(iv - 0.075):.2e}")

In [ ]:
# Validation Pydantic — 422 attendu
r = c.post("/api/v1/price", json={**atm, "volatility": 0})
print(f"volatility=0        → {r.status_code} (attendu 422)")

r = c.post("/api/v1/price", json={**atm, "option_type": "BLABLA"})
print(f"option_type bad     → {r.status_code} (attendu 422)")

# Implied vol hors bracket → 422
r = c.post("/api/v1/iv", json={**iv_req, "market_price": 2.0})
print(f"market_price=2.0    → {r.status_code} (attendu 422, vol hors [1e-6, 5.0])")
print(f"  detail            : {r.json().get('detail', '')[:80]}")

## 9. Cleanup

Fermeture propre du client httpx.

In [ ]:
c.close()
print("client httpx fermé")

## Cheat sheet — troubleshooting

| Erreur | Cause probable | Fix |
|---|---|---|
| `ConnectionError` sur localhost:8000 | uvicorn pas lancé | `.\scripts\run_api.ps1` |
| `No module named 'httpx'` | httpx pas dans venv | `python -m pip install httpx` |
| `redis: "DOWN"` | Redis pas up | `docker compose ... up -d redis` |
| `database: "DOWN"` | Postgres pas up ou mauvais URL | `docker compose ... up -d postgres` |
| `engines.*: "DOWN"` | `app.py` ne tourne pas | lancer `app.py` + Connect + Start Engine |
| `/iv` → 422 sur round-trip | bug bs_pricer F/K/T | investiger la formule BS |
| parity diff > 1e-6 | bs_price retourne des valeurs différentes sur C et P à l'ATM | bug sign dans bs_pricer |